In [1]:
import os, json, mimetypes
from pathlib import Path
from azure.storage.blob import BlobServiceClient, ContentSettings, ContainerClient
from uuid import uuid4
from pprint import pprint
from PIL import Image
from urllib.parse import urlparse
from pymongo import MongoClient, ReplaceOne
import bson

In [2]:
CONN_STR = os.getenv('AZURE_CONNECTION_STRING')
MONGO_URI = os.getenv('MONGO_DEFAULT_URI')
MONGO_DB_NAME = os.getenv('DEFAULT_DB_NAME')
COLLECTION = os.getenv('DEFAULT_COLLECTION_NAME')
CONTAINER = 'media'
SRC_FILES = ['image_description_date.json', 'single_option_data.json']

{
    'category_display_name': 'Jogi és igazgatási kérdések',
    'text': 'Kié hazánkban a vad tulajdonjoga?',
    'options': [
        {'id': 1, 'option': 'Vadászatra jogosult'},
        {'id': 2, 'option': 'Földtulajdonhoz kötött'},
        {'id': 3, 'option': 'Állam'}],
    'answer': 3,
    'category': 'JOGI_ES_IGAZGATASI_KERDESEK',
    'type': 'SINGLE_OPTION'
    }


In [3]:
def uploadImage(container: ContainerClient, image_path, id):
    blob_name = f'{id}{image_path.suffix}'
    blob = container.get_blob_client(blob_name)
    if not blob.exists():
        ctype = mimetypes.guess_type(image_path.name)[0] or "application/octet-stream"
        with image_path.open('rb') as fh:
            blob.upload_blob(fh,
                content_settings = ContentSettings(content_type=ctype)
            )
    return blob.url


In [ ]:
def buildDoc(question, container: ContainerClient):
    base = {
        "_id": str(uuid4()),
        "type": question['type'],
        "question": question.get("text", "")
    }

    type = question['type']

    if type == "SINGLE_OPTION":
        options = [{"id": str(uuid4()), "option": o['text']} for o in question['options']]
        correct = options[question['answer']-1]['id']
        
        return {
            **base,
            'category': question['category'],
            'category_display_name': question['category_display_name'],
            'options': options,
            'correct_option': correct 
        }

    if type == "IMAGE_DESCRIPTION":
        image_path = Path(f'data/{question['url']}')
        return {
            **base,
            'url': uploadImage(container, image_path, base["_id"]),
            'answer': question['answer']
        }

        


In [62]:
docs = []
blob_svc = BlobServiceClient.from_connection_string(CONN_STR)
container = blob_svc.get_container_client(CONTAINER)
try:
    container.create_container()
except Exception:
    pass
for source in os.listdir('data/questions'):
    src = json.loads(Path(f'data/questions/{source}').read_text())
    docs += [buildDoc(q, container) for q in src]

collection = MongoClient(MONGO_URI).get_database(MONGO_DB_NAME).get_collection(COLLECTION)
collection.bulk_write([ReplaceOne({'_id': d['_id']}, d, upsert=True) for d in docs])

BulkWriteResult({'writeErrors': [], 'writeConcernErrors': [], 'nInserted': 0, 'nUpserted': 698, 'nMatched': 0, 'nModified': 0, 'nRemoved': 0, 'upserted': [{'index': 0, '_id': '71f356d0-2aaa-4eab-910e-dabe21fb17ee'}, {'index': 1, '_id': '97f363a4-f5d2-4248-8bff-8542b9a31385'}, {'index': 2, '_id': 'e15419ad-b2ab-4aee-a00e-d1ab34a41763'}, {'index': 3, '_id': '3146d018-0047-445f-9e2b-75b078d6a5f3'}, {'index': 4, '_id': '5b56e093-2c96-4d46-8beb-792a1e6ca226'}, {'index': 5, '_id': '289ebb99-332b-49b5-902b-3a60e37d8ec3'}, {'index': 6, '_id': 'f7eb4923-83a5-4c05-9bc0-69a994591c74'}, {'index': 7, '_id': '18d9f7de-68aa-4f20-b874-54f64951e96f'}, {'index': 8, '_id': 'c50605b4-199f-451d-b85c-5fbbbc279666'}, {'index': 9, '_id': 'fdd0b0db-2fdc-4764-8bee-167e96eaa8ff'}, {'index': 10, '_id': 'fd2a2cde-9568-4d67-bd1f-eac4d2370bca'}, {'index': 11, '_id': '5ed79764-53a8-4e05-b4d9-5ed791e755b0'}, {'index': 12, '_id': '1d14f452-1669-4fd7-97c5-609264f4ae9f'}, {'index': 13, '_id': 'e141b8ba-cc35-4f0b-9004-225

In [67]:
len(collection.find(filter={'type': "SINGLE_OPTION", 'category': 'JOGI_ES_IGAZGATASI_KERDESEK'}).to_list())

32